In [ ]:
# 1. Partial clone (lazy blobs; 500MB OCR models never pulled for textlayer) + deps
!git clone --depth 1 --filter=blob:none https://github.com/aditya55496/eci-rolls.git /content/eci-rolls
%cd /content/eci-rolls
!pip -q install requests cryptography pymupdf pyarrow pandas

In [ ]:
# 2. Self-test gate: part-list -> 1 PDF -> parse -> count (no names). Confirms this IP can fetch PDFs.
!python -u uk/uk_driver.py check

In [ ]:
# 3. FULL 70-AC 2027 SSR pull -> LOCAL /content/out (reliable; Drive mount not needed yet).
#    PDFs streamed through RAM + discarded; only Parquet is kept. ~30-60 min.
!python -u uk/uk_driver.py textlayer --out /content/out --acs 1-70

In [ ]:
# 4. Statewide detection -> PII-safe flags.json (aggregate counts only, no names/EPICs).
!python -u uk/detect.py --data /content/out --out /content/out/flags.json

In [ ]:
# 5. Proof: record totals + headline signal (no names).
import glob, json, pyarrow.parquet as pq
files = sorted(glob.glob('/content/out/release/2027-ssr/ac=*/data.parquet'))
tot = sum(pq.ParquetFile(f).metadata.num_rows for f in files)
print('ACs with data:', len(files), '| total records:', tot)
fl = json.load(open('/content/out/flags.json'))
print('flags: acs=', fl['acs_analyzed'], '| cross-AC dup EPIC=', fl['dup_epic_cross_ac'])

In [ ]:
# 6. Copy to Google Drive (the private store) with mount retry (gotcha: mount is flaky).
#    Raw Parquet -> Drive/voter-engine/data/release (PRIVATE). flags.json -> shared/ (PII-safe).
import shutil, os, time
from google.colab import drive
mounted = False
for a in range(3):
    try:
        drive.mount('/content/drive', force_remount=True); mounted = True; break
    except Exception as e:
        print('mount retry', a, e); time.sleep(5)
if mounted:
    DEST = '/content/drive/MyDrive/voter-engine/data'
    os.makedirs(DEST, exist_ok=True)
    if os.path.isdir('/content/out/release'):
        shutil.copytree('/content/out/release', os.path.join(DEST, 'release'), dirs_exist_ok=True)
    sh = '/content/drive/MyDrive/voter-engine/shared'; os.makedirs(sh, exist_ok=True)
    shutil.copy('/content/out/flags.json', os.path.join(sh, 'uk_2027_flags.json'))
    print('COPIED to Drive:', DEST, '+ shared/uk_2027_flags.json')
else:
    print('Drive mount failed 3x -> files remain in /content/out; re-run this cell or download manually.')